# Two-group ROI analysis: learners (non-Mandarin) vs. native listeners (Mandarin)

Reads the per-ROI tables written by `extract_roi_values.py` (both groups, all contrasts, one first-level model: `scrubbed-splitfb`) and runs the analyses in `09_integration/TWO_GROUP_MANUSCRIPT_PLAN.md` (R2–R4). The statistics live in `group_stats.py`; this notebook is the sequence of calls plus figures.

**Before trusting anything here:** the learners' rows must reproduce `group_level_all_ROI.ipynb` (section 1b below).

Contrasts:
- `sound`, `resp`: vs. baseline, both groups
- `fbcorrect`: correct feedback vs. baseline, both groups -- the feedback contrast for group comparisons
- `fbcorrectvswrong`: learners only (natives get ~1 wrong-feedback trial per run, so it's missing or unreliable for them)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import group_stats as gs

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

nilearn_dir = '/ix1/bchandrasekaran/krs228/data/FLT/data_denoised/derivatives/nilearn'
group_dir = os.path.join(nilearn_dir, 'group_space-T1w_fwhm-0.00')
roi_dir = os.path.join(group_dir, 'roi-values_scrubbed-splitfb')
out_dir = os.path.join(group_dir, 'stats_two-group')
os.makedirs(out_dir, exist_ok=True)

# resampling settings, matching robustness_checks.py
n_perms, n_boot, seed = 10000, 10000, 0

## 1. Load and check

In [ ]:
df = gs.load_roi_values(roi_dir)
print(df.groupby('participant_group').participant_id.nunique())
gs.coverage(df)

Inputs `extract_roi_values.py` couldn't find. Expected: Mandarin participants' `fbcorrectvswrong` in stages where a run had no wrong trials. Anything else needs a look.

In [ ]:
missing_fpath = os.path.join(roi_dir, 'roi-values_missing.tsv')
try:
    missing = pd.read_csv(missing_fpath, sep='\t')
except pd.errors.EmptyDataError:   # nothing was missing
    missing = pd.DataFrame()
missing

### 1b. Learners reproduce `group_level_all_ROI.ipynb`

Compares the learners' rows with the tables `group_level_all_ROI.ipynb` saved (run it on `scrubbed-splitfb` first). Max difference should be ~0 for every network × contrast; IFGo/IFGt are dropped here, so the PFC row counts are lower.

In [ ]:
checks = []
for network in gs.NETWORKS:
    for contrast in ['sound', 'resp', 'fbcorrectvswrong']:
        nb_fpath = os.path.join(group_dir, f'univariate-results_network-{network}_contrast-{contrast}.tsv')
        if not os.path.exists(nb_fpath):
            continue
        nb_df = pd.read_csv(nb_fpath, sep='\t')
        mine = df[(df.network == network) & (df.contrast == contrast) & (df.participant_group == 'non-Mandarin')]
        merged = mine.merge(nb_df, on=['participant_id', 'region_hemi', 'learning_stage'], suffixes=('', '_nb'))
        checks.append({'network': network, 'contrast': contrast, 'rows_matched': len(merged),
                       'rows_here': len(mine), 'max_abs_diff': (merged.beta - merged.beta_nb).abs().max()})
pd.DataFrame(checks)

## 2. Responses vs. baseline within each group

Stages averaged, hemispheres averaged; FDR across regions within each group × contrast × network.

In [ ]:
baseline = gs.baseline_tests(df)
baseline.to_csv(os.path.join(out_dir, 'baseline_by_group.tsv'), sep='\t', index=False)

(baseline[baseline.contrast.isin(['sound', 'fbcorrect'])]
 .assign(result=lambda d: d.t.round(2).astype(str) + np.where(d.q < .05, '*', ''))
 .pivot_table(index=['network', 'region'], columns=['contrast', 'participant_group'], values='result', aggfunc='first'))

## 3. Does `fbcorrect` stand in for `fbcorrectvswrong` in learners?

The group comparison of feedback has to use `fbcorrect`. It's only a fair stand-in if, in learners, it shows the same learning-stage trend as the headline `fbcorrectvswrong` effect (plan R3). If it doesn't, section 4's feedback result needs reframing before it's interpreted.

In [ ]:
trends = gs.subject_trends(df)
trends.to_csv(os.path.join(out_dir, 'subject_trends.tsv'), sep='\t', index=False)

pd.concat([gs.learner_contrast_agreement(trends, region) for region in ['aCAU', 'aPUT']]).round(3)

## 4. Planned test: is the aCAU feedback decline learner-specific?

Per-subject learning-stage trend (`[-1, 0, 1]`, final − early), within each group and learners − natives: Welch t, group-label permutation, leave-one-out, bootstrap CI. Then the same for the aCAU − aPUT differential trend.

In [ ]:
planned = pd.concat([
    gs.trend_group_difference(trends, 'fbcorrect', 'aCAU', n_perms=n_perms, n_boot=n_boot, seed=seed),
    gs.trend_group_difference(gs.differential_trend(trends, 'fbcorrect', 'aCAU', 'aPUT'),
                              'fbcorrect', 'aCAU-minus-aPUT', n_perms=n_perms, n_boot=n_boot, seed=seed),
], ignore_index=True)
planned.to_csv(os.path.join(out_dir, 'planned_trend_group_tests.tsv'), sep='\t', index=False)
planned.round(4)

## 5. Group × stage, every region

Mixed ANOVA per region (stage within, group between), FDR per contrast × effect across all regions. Exploratory beyond the planned test in section 4.

In [ ]:
anova = gs.group_by_stage_anova(df, contrasts=['sound', 'fbcorrect', 'resp'])
anova.to_csv(os.path.join(out_dir, 'group_by_stage_anova.tsv'), sep='\t', index=False)
anova[anova.q < .05].sort_values(['contrast', 'effect', 'q']).round(4)

## 6. Figure: stage profiles by group

In [ ]:
bilat = gs.bilateral_mean(df)
panels = [('fbcorrect', 'aCAU'), ('fbcorrect', 'aPUT'), ('sound', 'aCAU'), ('sound', 'pPUT')]
colors = {'non-Mandarin': '#2a6fdb', 'Mandarin': '#d9822b'}

fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.2), sharey=False)
for ax, (contrast, region) in zip(axes, panels):
    sub = bilat[(bilat.contrast == contrast) & (bilat.region == region)]
    for group in gs.GROUPS:
        g = sub[sub.participant_group == group].groupby('learning_stage').beta
        mean, sem = g.mean().reindex(gs.STAGE_ORDER), g.sem().reindex(gs.STAGE_ORDER)
        ax.errorbar(range(3), mean, yerr=sem, marker='o', capsize=3, color=colors[group],
                    label='learners' if group == 'non-Mandarin' else 'natives')
    ax.axhline(0, color='0.6', lw=0.8)
    ax.set_xticks(range(3), gs.STAGE_ORDER)
    ax.set_title(f'{region} · {contrast}')
axes[0].set_ylabel('effect size (mean ± SEM)')
axes[0].legend(frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'stage_profiles_by_group.png'), dpi=300)

## 7. Learners only: headline `fbcorrectvswrong`

For reference alongside section 4; values match `robustness_summary.csv`.

In [ ]:
learners = trends[trends.participant_group == 'non-Mandarin']
rows = []
for label, values in [('aCAU', learners.query("contrast == 'fbcorrectvswrong' and region == 'aCAU'").trend),
                      ('aCAU-minus-aPUT', gs.differential_trend(trends, 'fbcorrectvswrong', 'aCAU', 'aPUT')
                                            .query("participant_group == 'non-Mandarin'").trend)]:
    loso = gs.leave_one_out_stability(values)
    rows.append({'trend': label, 'n': len(values), 'mean': values.mean(),
                 't': loso['original_t'], 'p': loso['original_p'],
                 'exact_permutation_p': gs.exact_sign_flip_permutation_test(values)['p_value'],
                 'loso_preserved': f"{loso['n_preserved']}/{loso['n_folds']}"})
pd.DataFrame(rows).round(4)